# 08 — Lag Feature Engineering & Retrain (Action 2.3)

**Prerequisite**: `notebooks/03_clustering.ipynb` (or existing zone_hour_grid.parquet) must be complete.  
These files must exist:
- `data/processed/zone_hour_grid.parquet` — will be **regenerated** with lag features
- `data/processed/zone_day_grid.parquet` — will be **regenerated** with lag features
- `data/processed/cis_table.parquet`
- `data/processed/features_with_zones.parquet` — output of 03_clustering.ipynb

**What happens here**:
1. Regenerate zone grids with `lag_1d_count` + `lag_7d_count` using `aggregate_to_zone_grid()`
2. Verify leakage-free properties of both lag columns
3. Retrain all models (XGBoost, LightGBM, CatBoost) on hour resolution with the new features
4. Compare against the v2.1 baseline (CatBoost, MAE=4.5863, per-hour NDCG=0.8888)
5. Update `configs/model.yaml` if the new winner is better

**Features.yaml version**: v2.2 (lag_1d_count + lag_7d_count added)

**Files saved**:
- `data/processed/zone_hour_grid.parquet` — overwritten with 2 new columns
- `data/processed/zone_day_grid.parquet` — overwritten with 2 new columns
- `checkpoints/{model}_hour_{timestamp}/` — new checkpoints for v2.2 run
- `data/outputs/eval_{timestamp}.json` — full evaluation results

## Cell 1 — Environment setup
**Expected output**: `Project root: ...GridLock_R2_Transfer`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\USER\Desktop\GridLock_R2_Transfer


## Cell 2 — Configure loguru
**Expected output**: `Loguru configured.`

In [2]:
import sys
from loguru import logger

logger.remove()
logger.add(
    sys.stdout,
    format='<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}',
    level='INFO',
    colorize=False,
)
print('Loguru configured.')

Loguru configured.


## Cell 3 — Verify prerequisite files & confirm features.yaml version
**What this cell does**: Check that grids exist and features.yaml is now v2.2 with lag features listed.  
**Expected output**: All files found. `features.yaml version: 2.2`. Both lag features listed.

In [3]:
import yaml

required_files = [
    PROJECT_ROOT / 'data' / 'processed' / 'features_with_zones.parquet',
    PROJECT_ROOT / 'data' / 'processed' / 'cis_table.parquet',
    PROJECT_ROOT / 'configs' / 'features.yaml',
]

all_ok = True
for f in required_files:
    if f.exists():
        print(f'  ✓  {f.name}  ({f.stat().st_size / 1e3:.1f} KB)')
    else:
        print(f'  ✗  MISSING: {f.name}')
        all_ok = False

# Verify features.yaml v2.2 contains lag features
with open(PROJECT_ROOT / 'configs' / 'features.yaml') as f:
    feat_cfg = yaml.safe_load(f)

version = feat_cfg.get('version', '?')
historical = feat_cfg.get('historical', [])
print(f'\nfeatures.yaml version: {version}')
print(f'Historical features: {historical}')

if 'lag_1d_count' not in historical or 'lag_7d_count' not in historical:
    print('  ✗  lag_1d_count or lag_7d_count not found in features.yaml historical section!')
    all_ok = False
else:
    print('  ✓  lag_1d_count + lag_7d_count present in features.yaml')

if not all_ok:
    raise FileNotFoundError('One or more prerequisites are missing. See above.')
print('\nAll prerequisite checks passed. Proceed.')

  ✓  features_with_zones.parquet  (8444.2 KB)
  ✓  cis_table.parquet  (10.7 KB)
  ✓  features.yaml  (10.7 KB)

features.yaml version: 2.2b
Historical features: ['rolling_7d_count', 'lag_1d_count', 'lag_7d_count']
  ✓  lag_1d_count + lag_7d_count present in features.yaml

All prerequisite checks passed. Proceed.


## Cell 4 — Regenerate zone grids with lag features
**What this cell does**: Load `features_with_zones.parquet` and call `aggregate_to_zone_grid()` for
both hour and day resolutions. This will produce two new columns (`lag_1d_count`, `lag_7d_count`)
alongside the existing `rolling_7d_count`.

**Why regenerate**: The existing grids were produced by v2.1 `aggregate_to_zone_grid()` which did
not include lag features. The v2.2 function now computes them. The grids are overwritten in-place.

**Expected output**: `✓ Aggregation complete (hour)` and `(day)` with lag_1d and lag_7d non-zero counts.

**Expected runtime**: ~30–60 seconds.

In [4]:
import pandas as pd
from src.data.features import aggregate_to_zone_grid

# Load feature-engineered + zone-assigned DataFrame
feat_path = PROJECT_ROOT / 'data' / 'processed' / 'features_with_zones.parquet'
df = pd.read_parquet(feat_path)
print(f'Loaded features_with_zones.parquet: {df.shape}')
print(f'  zone_id unique: {df["zone_id"].nunique()}')
print(f'  date range: {df["created_datetime"].min()} → {df["created_datetime"].max()}')
print()

# Regenerate hour grid
print('Regenerating zone_hour_grid.parquet with lag features...')
hour_grid = aggregate_to_zone_grid(
    df,
    time_resolution='hour',
    save_path=PROJECT_ROOT / 'data' / 'processed' / 'zone_hour_grid.parquet',
)

# Regenerate day grid
print()
print('Regenerating zone_day_grid.parquet with lag features...')
day_grid = aggregate_to_zone_grid(
    df,
    time_resolution='day',
    save_path=PROJECT_ROOT / 'data' / 'processed' / 'zone_day_grid.parquet',
)
print()
print('Grid regeneration complete.')

Loaded features_with_zones.parquet: (268281, 23)
  zone_id unique: 140
  date range: 2023-11-09 19:11:46+00:00 → 2024-04-08 17:30:46+00:00

Regenerating zone_hour_grid.parquet with lag features...
22:34:12 | INFO     | Aggregating to zone × hour grid ...


Aggregating (zone×hour): 100%|██████████| 4/4 [00:11<00:00,  2.97s/step]

22:34:23 | INFO     | ✓ Aggregation complete (hour): 26,354 zone×time rows | 140 unique zones | target='zone_hour_violation_count' (mean=10.18, max=284) | rolling_7d non-zero: 24,498/26,354 | lag_1d non-zero: 10,504/26,354 | lag_7d non-zero: 10,179/26,354
22:34:23 | INFO     | Aggregated grid saved → 'C:\Users\USER\Desktop\GridLock_R2_Transfer\data\processed\zone_hour_grid.parquet'

Regenerating zone_day_grid.parquet with lag features...
22:34:23 | INFO     | Aggregating to zone × day grid ...



Aggregating (zone×day): 100%|██████████| 4/4 [00:03<00:00,  1.10step/s]

22:34:27 | INFO     | ✓ Aggregation complete (day): 8,246 zone×time rows | 140 unique zones | target='zone_day_violation_count' (mean=32.53, max=1848) | rolling_7d non-zero: 8,106/8,246 | lag_1d non-zero: 5,269/8,246 | lag_7d non-zero: 4,817/8,246
22:34:27 | INFO     | Aggregated grid saved → 'C:\Users\USER\Desktop\GridLock_R2_Transfer\data\processed\zone_day_grid.parquet'

Grid regeneration complete.


## Cell 5 — Verify lag columns: spot-check leakage-free property
**What this cell does**: Assert that for every `(zone_id, hour_of_day)` group,
`lag_1d_count[t]` equals the actual target on the immediately preceding date (or 0 if no prior day).
Also prints descriptive stats for both lag columns.

**Expected output**: Verification PASSED. Stats showing lag columns are non-zero for most rows.

In [5]:
import numpy as np
import pandas as pd

# Reload from disk to test the saved file
hour_grid = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'zone_hour_grid.parquet')

print(f'zone_hour_grid.parquet shape: {hour_grid.shape}')
print(f'Columns: {list(hour_grid.columns)}')
print()

# Check lag features exist
for col in ['rolling_7d_count', 'lag_1d_count', 'lag_7d_count']:
    assert col in hour_grid.columns, f'MISSING: {col}'
    non_zero = (hour_grid[col] > 0).sum()
    print(f'  {col}: non-zero = {non_zero:,}/{len(hour_grid):,} ({non_zero/len(hour_grid)*100:.1f}%)'
          f'  |  mean={hour_grid[col].mean():.3f}  max={hour_grid[col].max():.0f}')

print()

# Spot-check: for zone 2 (busiest zone), verify lag_1d_count[t] = target[t-1]
zone2 = hour_grid[hour_grid['zone_id'] == 2].sort_values(['hour_of_day', 'date'])
if len(zone2) > 0:
    # Pick hour_of_day=9 (busiest)
    z2h9 = zone2[zone2['hour_of_day'] == 9].reset_index(drop=True)
    if len(z2h9) >= 3:
        target_col = 'zone_hour_violation_count'
        # lag_1d_count at index i should equal target at index i-1
        errors = 0
        for i in range(1, min(10, len(z2h9))):
            expected = float(z2h9.loc[i-1, target_col])
            actual   = float(z2h9.loc[i, 'lag_1d_count'])
            if abs(expected - actual) > 1e-3:
                print(f'  MISMATCH at i={i}: expected lag_1d={expected}, got {actual}')
                errors += 1
        if errors == 0:
            print(f'  ✓  lag_1d_count leakage check PASSED (zone_id=2, hour=9, {min(9, len(z2h9)-1)} rows checked)')
        else:
            print(f'  ✗  lag_1d_count leakage check FAILED — {errors} mismatches!')
    else:
        print('  zone_id=2 hour=9 has <3 rows — skipping spot-check')
else:
    print('  zone_id=2 not found — skipping spot-check')

print()
print('Verification complete.')

zone_hour_grid.parquet shape: (26354, 18)
Columns: ['zone_id', 'date', 'hour_of_day', 'zone_hour_violation_count', 'fraction_at_junction', 'dominant_violation_type', 'dominant_vehicle_type', 'violation_type_primary_encoded', 'vehicle_type_encoded', 'police_station_id', 'center_code_encoded', 'data_sent_to_scita_mean', 'is_weekend', 'day_of_week', 'month', 'rolling_7d_count', 'lag_1d_count', 'lag_7d_count']

  rolling_7d_count: non-zero = 24,498/26,354 (93.0%)  |  mean=9.907  max=194
  lag_1d_count: non-zero = 10,504/26,354 (39.9%)  |  mean=7.917  max=284
  lag_7d_count: non-zero = 10,179/26,354 (38.6%)  |  mean=7.594  max=284

  MISMATCH at i=9: expected lag_1d=19.0, got 0.0
  ✗  lag_1d_count leakage check FAILED — 1 mismatches!

Verification complete.


## Cell 6 — Retrain all models on v2.2 features
**What this cell does**: Calls `src/training/train.run_training()` — the same full 6-model run as
`04_training.ipynb` but now with 20 features (18 v2.1 + lag_1d_count + lag_7d_count).

The run:
- Splits: train ≤ 2024-02-29, test ≥ 2024-03-01 (same as always)
- Trains XGBoost, LightGBM, CatBoost on hour + day (6 runs)
- Evaluates all metrics including per-hour NDCG, Spearman ρ, PAI
- Saves all checkpoints and updates `model.yaml` winner

**Expected runtime**: ~5–10 minutes

**Expected output**: Scorecard for each model + final comparison table + winner announcement.

In [6]:
from src.training.train import run_training

results = run_training(project_root=PROJECT_ROOT)
print('\nTraining complete.')

22:34:27 | INFO     | Configs loaded | features.yaml hash: 68878b732a7d6fdf...
22:34:27 | INFO     | Training run started | timestamp=20260619_170427 | seed=42
22:34:27 | INFO     | CIS table loaded: 140 zones


Training all models:   0%|          | 0/6 [00:00<?, ?run/s]

22:34:28 | INFO     | 
  Training: XGBOOST | resolution=hour
  Target: zone_hour_violation_count | Features: 20
22:34:28 | INFO     | Split: train=19,870 rows (2023-11-09 → 2024-02-29) | test=6,484 rows (2024-03-01 → 2024-04-08) | Leakage guard: PASSED ✓
22:34:28 | INFO     | Computing zone aggregate features (train-only) ...
22:34:28 | INFO     | Zone aggregates computed for 140 training zones | zone_mean_count range: [1.10, 65.32]
22:34:28 | INFO     | X_train: (19870, 20)  y_train mean=10.30 max=284
22:34:28 | INFO     | X_val:   (6484, 20)    y_val mean=9.82 max=266



Fitting xgboost (hour):   0%|          | 0/1 [00:00<?, ?model/s]

22:34:29 | INFO     |   XGBoost early-stop: best_round=63 | final_val_rmse=10.3278


...

22:35:02 | INFO     |   CatBoost early-stop: best_round=82 | final_val_rmse=27.6838



Fitting catboost (day): 100%|██████████| 1/1 [00:00<00:00,  2.89model/s]
                                                                        

22:35:02 | INFO     | === Evaluating: catboost | day | target='zone_day_violation_count' ===
22:35:02 | INFO     | [catboost/day] MAE=10.7470  RMSE=27.5827
22:35:02 | INFO     | Naive mean-per-zone baseline: MAE=11.7889  RMSE=36.2259
22:35:02 | INFO     |   BEATS naive baseline: MAE 10.7470 vs 11.7889 (+8.8% improvement)
22:35:02 | INFO     | Relevance assigned: grade=2 → 35 zones | grade=1 → 35 zones | grade=0 → 67 zones (q50=36.0, q75=113.0)
22:35:02 | INFO     |   [catboost] NDCG@5=1.0000  Precision@5=1.0000
22:35:02 | INFO     |   [catboost] NDCG@10=1.0000  Precision@10=1.0000
22:35:02 | INFO     | Frequency baseline: 140 zones | top-3 zones: {2: 185578.5, 7: 1537.732546, 4: 160.647921}
22:35:02 | INFO     |   [freq-baseline] NDCG@5=1.0000  Precision@5=1.0000
22:35:02 | INFO     |   [freq-baseline] NDCG@10=1.0000  Precision@10=1.0000
22:35:02 | WARNING  |   [catboost] does NOT beat freq ranker on NDCG.
22:35:02 | INFO     |   [catboost] Computing per-hour ranking metrics ...
22:35:

Training all models: 100%|██████████| 6/6 [00:34<00:00,  5.83s/run]

22:35:02 | INFO     | 
  MODEL COMPARISON SUMMARY
22:35:02 | INFO     |   xgboost_hour                   | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=4.6783
22:35:02 | INFO     |   lightgbm_hour                  | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=4.6653
22:35:02 | INFO     |   catboost_hour                  | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=4.8235
22:35:02 | INFO     |   xgboost_day                    | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=9.8116
22:35:02 | INFO     |   lightgbm_day                   | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=9.9239
22:35:02 | INFO     |   catboost_day                   | NDCG@10=1.0000 | Prec@10=1.0000 | MAE=10.7470
22:35:02 | INFO     | 
  🏆 WINNER: lightgbm_hour | NDCG@10=1.0000 | Precision@10=1.0000 | MAE=4.6653
22:35:02 | INFO     | Eval results saved → 'C:\Users\USER\Desktop\GridLock_R2_Transfer\data\outputs\eval_20260619_170427.json'
22:35:03 | INFO     | model.yaml winner section updated → 'C:\Users\USER\Desktop\GridLock_R2_Transfer\configs

## Cell 7 — Compare v2.2 winner vs v2.1 baseline
**What this cell does**: Extract winner metrics from this run and compare against the v2.1
baseline (CatBoost RMSE, MAE=4.5863, RMSE=10.1618, per-hour NDCG=0.8888).

**Expected output**: Delta table. If lag features helped, MAE and RMSE should be lower.

In [7]:
import pandas as pd

# v2.1 baseline (CatBoost, no lag features)
baseline_v21 = {
    'model':          'catboost_hour (v2.1)',
    'MAE':            4.5863,
    'RMSE':           10.1618,
    'Per-hour NDCG':  0.8888,
    'Spearman rho':   0.5123,
    'PAI':            11.19,
}

# v2.2 winner from this run
winner = results.get('winner', {})
winner_key = f"{winner.get('model', '?')}_{winner.get('resolution', '?')}"
winner_result = results.get(winner_key, {})
reg   = winner_result.get('regression', {})
rph   = winner_result.get('ranking_per_hour', {})
ndcg  = rph.get('model_ndcg', {})
spear = rph.get('model_spearman', {})
pai   = winner_result.get('spatial_pai', {})

v22_winner = {
    'model':          f"{winner_key} (v2.2 + lag features)",
    'MAE':            round(reg.get('mae', float('nan')), 4),
    'RMSE':           round(reg.get('rmse', float('nan')), 4),
    'Per-hour NDCG':  round(ndcg.get('mean_ndcg', float('nan')), 4),
    'Spearman rho':   round(spear.get('mean_spearman', float('nan')), 4),
    'PAI':            round(pai.get('pai', float('nan')), 2),
}

df_cmp = pd.DataFrame([baseline_v21, v22_winner]).set_index('model')

print('=== v2.1 Baseline vs v2.2 Winner ===')
pd.set_option('display.float_format', '{:.4f}'.format)
display(df_cmp)

print()
print('=== Delta (v2.2 - v2.1) ===')
numeric_cols = ['MAE', 'RMSE', 'Per-hour NDCG', 'Spearman rho', 'PAI']
for col in numeric_cols:
    delta = v22_winner[col] - baseline_v21[col]
    direction = '✓ BETTER' if (col in ('Per-hour NDCG', 'Spearman rho', 'PAI') and delta > 0) \
                           or (col in ('MAE', 'RMSE') and delta < 0) \
                else ('= SAME' if abs(delta) < 0.001 else '✗ WORSE')
    print(f'  {col:<20}: {delta:+.4f}  {direction}')

=== v2.1 Baseline vs v2.2 Winner ===


,MAE,RMSE,Per-hour NDCG,Spearman rho,PAI
model,,,,,
catboost_hour (v2.1),4.5863,10.1618,0.8888,0.5123,11.1900
lightgbm_hour (v2.2 + lag features),4.6653,10.4104,0.8871,0.5041,11.1900



=== Delta (v2.2 - v2.1) ===
  MAE                 : +0.0790  ✗ WORSE
  RMSE                : +0.2486  ✗ WORSE
  Per-hour NDCG       : -0.0017  ✗ WORSE
  Spearman rho        : -0.0082  ✗ WORSE
  PAI                 : +0.0000  = SAME


## Cell 8 — Print comparison table (all models)
**What this cell does**: Display the full 6-model comparison from this run, sorted by NDCG@10 / MAE.  
**Expected output**: Table of all 6 models. CatBoost hour should still be near the top.

In [8]:
import pandas as pd

rows = results.get('comparison_table', [])
if rows:
    df_full = pd.DataFrame(rows)
    display_cols = ['run', 'model', 'resolution', 'NDCG@10', 'Precision@10', 'MAE', 'RMSE', 'beats_baseline']
    display_cols = [c for c in display_cols if c in df_full.columns]
    print('=== Full Model Comparison (v2.2 — lag features) ===')
    pd.set_option('display.float_format', '{:.4f}'.format)
    display(df_full[display_cols])
else:
    print('No comparison table found in results dict.')

=== Full Model Comparison (v2.2 — lag features) ===


,run,model,resolution,NDCG@10,Precision@10,MAE,RMSE,beats_baseline
0,lightgbm_hour,lightgbm,hour,1.0000,1.0000,4.6653,10.4104,"{'ndcg': False, 'precision': False}"
1,xgboost_hour,xgboost,hour,1.0000,1.0000,4.6783,10.3178,"{'ndcg': False, 'precision': False}"
2,catboost_hour,catboost,hour,1.0000,1.0000,4.8235,10.3363,"{'ndcg': False, 'precision': False}"
3,xgboost_day,xgboost,day,1.0000,1.0000,9.8116,25.5519,"{'ndcg': False, 'precision': False}"
4,lightgbm_day,lightgbm,day,1.0000,1.0000,9.9239,25.4943,"{'ndcg': False, 'precision': False}"
5,catboost_day,catboost,day,1.0000,1.0000,10.7470,27.5827,"{'ndcg': False, 'precision': False}"


## Cell 9 — Summary
**What this cell does**: Print a final summary of what was done, what was saved, and next steps.  
**Expected output**: Paths to new checkpoints, delta table summary, next action.

In [9]:
import glob

ts = results.get('training_timestamp', 'unknown')
winner = results.get('winner', {})

print('=' * 60)
print('  LAG FEATURE RETRAIN SUMMARY')
print('=' * 60)
print()
print('What was done:')
print('  - Added lag_1d_count + lag_7d_count to features.yaml v2.2')
print('  - Regenerated zone_hour_grid.parquet + zone_day_grid.parquet')
print('  - Retrained XGBoost, LightGBM, CatBoost on 20 features')
print()
print('Files saved/updated:')
print(f'  - data/processed/zone_hour_grid.parquet (now has lag_1d_count, lag_7d_count)')
print(f'  - data/processed/zone_day_grid.parquet (now has lag_1d_count, lag_7d_count)')
eval_files = sorted(glob.glob(str(PROJECT_ROOT / 'data' / 'outputs' / 'eval_*.json')))
if eval_files:
    print(f'  - {Path(eval_files[-1]).name}')
for model_name in ['xgboost', 'lightgbm', 'catboost']:
    ckpt_dirs = sorted(glob.glob(str(PROJECT_ROOT / 'checkpoints' / f'{model_name}_hour_{ts}')))
    for d in ckpt_dirs:
        print(f'  - checkpoints/{Path(d).name}/')
print()
print('Winner:')
print(f'  {winner.get("run", "?")} | NDCG@10={winner.get("NDCG@10", "?"):.4f} | MAE={winner.get("MAE", "?"):.4f}')
print()
print('Next steps:')
print('  → If winner improved over v2.1 baseline (MAE < 4.5863):')
print('    Update session_log.md with the new metrics.')
print('  → Run 06_shap.ipynb on the new winning checkpoint (CB-3: CatBoost SHAP).')
print('  → After SHAP: run inference/static_output.py to regenerate the demo HTML.')

  LAG FEATURE RETRAIN SUMMARY

What was done:
  - Added lag_1d_count + lag_7d_count to features.yaml v2.2
  - Regenerated zone_hour_grid.parquet + zone_day_grid.parquet
  - Retrained XGBoost, LightGBM, CatBoost on 20 features

Files saved/updated:
  - data/processed/zone_hour_grid.parquet (now has lag_1d_count, lag_7d_count)
  - data/processed/zone_day_grid.parquet (now has lag_1d_count, lag_7d_count)
  - eval_20260619_170427.json
  - checkpoints/xgboost_hour_20260619_170427/
  - checkpoints/lightgbm_hour_20260619_170427/
  - checkpoints/catboost_hour_20260619_170427/

Winner:
  lightgbm_hour | NDCG@10=1.0000 | MAE=4.6653

Next steps:
  → If winner improved over v2.1 baseline (MAE < 4.5863):
    Update session_log.md with the new metrics.
  → Run 06_shap.ipynb on the new winning checkpoint (CB-3: CatBoost SHAP).
  → After SHAP: run inference/static_output.py to regenerate the demo HTML.
